# 2. Run the NLP on every head CT report

**What it does:** runs a fixed snapshot of the SDH NLP (`phenotype/nlp/NLP_v5_snapshot.ipynb`, see
`nlp/SNAPSHOT.md`) on every report from notebook 1, exactly as `NLP_v5.ipynb` processes a report
(tokenize, bleed tree, variable mapping). It is the pre-validation version, used to set up the
pipeline; when the validated NLP is ready, swap the snapshot and rerun from here.

A CT counts as **showing an SDH** when the NLP gives a laterality of Left / Right / Bilateral,
or any falx (parafalcine) or tentorium hematoma.

**Output:** `patient_level/ct_reports_nlp_results.csv` (report id, episode, every NLP variable,
error; no report text).

**Time:** test run about a minute; full run (~46,000 reports) roughly 30-60 minutes. Progress is
saved as it goes: if it stops, run the same cell again and it continues where it left off.
For the full run you can also use a terminal inside `tmux` (see the last cell).

> **Privacy:** everything this notebook *displays* comes from `aggregate/` (counts with cells under 6 suppressed). Files in `patient_level/` stay on GEMINI: don't print their rows, and never paste them outside GEMINI.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# To write the output somewhere else, uncomment and edit BEFORE importing sdh_config:
# os.environ["SDH_OUTPUT_DIR"] = str(Path.home() / "sdh_phenotype_output")

PHENO = Path.cwd().parent              # the phenotype/ folder (this notebook is in phenotype/notebooks/)
sys.path.insert(0, str(PHENO))
import sdh_config as cfg
import sdh_io as io
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 220)
AGG = cfg.AGGREGATE_DIR


def show(name, rows=None, note=None):
    """Display one aggregate (cell-suppressed) table from the output folder."""
    path = AGG / name
    if not path.exists():
        print(f"{name}: not there yet (run the step above first)")
        return None
    df = pd.read_csv(path)
    display(Markdown(f"**{name}**" + (f" - {note}" if note else "")))
    display(df.head(rows) if rows else df)
    return df


def share(names, rows=80):
    """Print aggregate tables as text to paste to Claude / your team (counts only, small cells suppressed)."""
    for name in names:
        path = AGG / name
        if path.exists():
            df = pd.read_csv(path)
            print(f"===== {name} ({len(df)} rows{', first ' + str(rows) if len(df) > rows else ''}) =====")
            print(df.head(rows).to_csv(index=False))
        else:
            print(f"===== {name}: not found =====")

## Test run (first 200 reports)

In [ ]:
TEST_OUT = cfg.PATIENT_DIR / "nlp_test_200.csv"
!{sys.executable} ../nlp/run_nlp.py --limit 200 --workers 2 --output "{TEST_OUT}"

In [ ]:
def nlp_summary(path):
    r = pd.read_csv(path, dtype=str, keep_default_na=False)
    lat = r.laterality.str.strip()
    comp = pd.Series(False, index=r.index)
    for c in cfg.NLP_SDH_COMPARTMENTS:
        comp |= ~r[c].str.strip().isin(cfg.NLP_ABSENT_VALUES)
    sdh = lat.isin(cfg.NLP_SDH_LATERALITY) | comp
    print(f"reports: {len(r)} | errors: {(r.error != '').sum()} | show an SDH: {sdh.sum()} ({100 * sdh.mean():.1f}%)")
    print(f"  SDH found only through a falx / tentorium hematoma: {(sdh & ~lat.isin(cfg.NLP_SDH_LATERALITY)).sum()}")
    display(lat.value_counts().rename("reports").to_frame().map(io.suppress))
    return r

test = nlp_summary(TEST_OUT)

**Look for:** errors close to 0; a sensible share showing an SDH (follow-up and unrelated head CTs
will often be negative). If something looks off, stop here and send me the summary.

## Full run

In [ ]:
!{sys.executable} ../nlp/run_nlp.py --workers 4

In [ ]:
full = nlp_summary(cfg.NLP_RESULTS_FILE)
for c in ["L_acuity", "R_acuity", "midline_shift_present", "mass_effect_present"]:
    if c in full:
        display(full[c].value_counts().rename(f"{c}: reports").to_frame().map(io.suppress))

## Share
The cells above print counts only. Paste their output (not the CSV) to Claude.

**Terminal alternative for the full run** (keeps going if the browser closes):
```bash
tmux new -s nlp
srun -p cpu --qos=depri --mem=32G --cpus-per-task=4 --pty bash -i
source ~/.venvs/sdh-gemini/bin/activate && cd ~/SDH_NLP
python phenotype/nlp/run_nlp.py --workers 4
```